In [29]:
import wikipediaapi
import sys
from collections import deque
import re
import json

In [23]:
wiki_wiki = wikipediaapi.Wikipedia(user_agent='LeChatBot (academic research)', 
                                   language='en',
                                   extract_format=wikipediaapi.ExtractFormat.WIKI)


In [24]:
page_py = wiki_wiki.page('National Basketball Association')


In [25]:
KEYWORDS = [
    "nba", "basketball", "season", "player", "team",
    "draft", "playoffs", "coach", "finals"
]

def is_relevant(title):
    t = title.lower()
    return any(k in t for k in KEYWORDS)

In [26]:
#BFS crawler for NBA Wikipedia pages
def crawl_wikipedia(seed_titles, max_depth=2, max_pages=500):
    visited = set()
    queue = deque([(title, 0) for title in seed_titles])
    corpus = {}

    while queue and len(corpus) < max_pages:
        title, depth = queue.popleft()

        if title in visited or depth > max_depth:
            continue

        page = wiki_wiki.page(title)
        if not page.exists() or len(page.text) < 500:
            continue

        visited.add(title)
        corpus[title] = page.text

        # Explore links
        for linked_title in page.links.keys():
            if linked_title not in visited and is_relevant(linked_title):
                queue.append((linked_title, depth + 1))

    return corpus

In [27]:
def clean_text(text):
    # Basic cleaning: remove extra whitespace and non-ASCII characters
    text = ' '.join(text.split())
    text = ''.join(c for c in text if ord(c) < 128)
    
    return text

In [28]:
seed_pages = [
    "National Basketball Association",
    "NBA playoffs",
    "History of the NBA",
    "List of NBA teams",
    "List of NBA players"
]

nba_corpus = crawl_wikipedia(seed_pages, max_depth=2, max_pages=500)
nba_corpus = {k: clean_text(v) for k, v in nba_corpus.items()}


In [30]:
with open("nba_wikipedia_corpus.json", "w") as f:
    json.dump(nba_corpus, f, indent=2)